In [1]:
%matplotlib tk
import torch
import torch.nn.functional as F
import os
import scipy as sc
import numpy as np
import sympy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import scienceplots
import matplotlib.animation as animation
from scipy.ndimage import binary_dilation

#plt.style.use(['science','notebook', 'grid'])
device = torch.device('cuda' if torch.cuda.is_available( ) else 'cpu')

In [2]:
def Gauss_Siedel_relaxation(Vi, mask, max_iter=100):
    V = Vi.clone()
    for _ in range(max_iter):

        V = 0.25 * (V.roll(1, 0) + V.roll(-1, 0) +
                          V.roll(1, 1) + V.roll(-1, 1))
        V[mask] = Vi[mask] 
    return V

In [3]:
dis_bnd = torch.ones((10,10), dtype=torch.bool)
dis_bnd[0, :] = False
dis_bnd[-1, :] = False
dis_bnd[:, 0] = False
dis_bnd[:, -1] = False
plt.imshow(dis_bnd, cmap='gray')

In [4]:
N = 100
potential = torch.zeros((N, N), device=device)
potential[15:27, 15:27] = 1.0  # A square region with potential
potential[0,:] = 2
mask = potential > 0
plt.imshow(mask.cpu().numpy(), cmap='viridis', interpolation='nearest')

potential = Gauss_Siedel_relaxation(potential, mask, max_iter=1000)

In [5]:
plt.imshow(potential.cpu().numpy(), cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title('Electric Potential')

Text(0.5, 1.0, 'Electric Potential')

In [6]:
Ex = -torch.gradient(potential, dim=1)[0]
Ey = -torch.gradient(potential, dim=0)[0]

plt.quiver(Ex.cpu().numpy(), Ey.cpu().numpy(), color='blue')
plt.imshow(potential.cpu().numpy(), cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title('Electric Potential')

Text(0.5, 1.0, 'Electric Potential')

In [29]:
def xy_to_index(x, y, grid_size):
    """Convert (x, y) coordinates to grid indices."""
    i = int(x * grid_size)
    j = int(y * grid_size)
    return i, j

def swap(x, y):
    temp = x
    x = y
    y = temp
    return x, y

In [32]:
NUM_ITERATIONS = 10000
dt = 0.01
m = 1
q = 1
x0 = 0.1; y0 = 0.5
x1 = 0.1; y1 = 0.5
x = [x0, x1]; y = [y0, y1]
t = np.arange(0, 10, dt)

In [33]:
for k in range(NUM_ITERATIONS-2):
    i, j = xy_to_index(x[k+1], y[k+1], N)
    ax = q * Ex[i,j].cpu()/ m
    ay = q * Ey[i,j].cpu() / m

    x = np.append(x, 2 * x[k+1] - x[k] + ax * dt**2)
    y = np.append(y, 2 * y[k+1] - y[k] + ay * dt**2)
    if x[k+2] > 1 or x[k+2] < 0:
        x[k+1], x[k+2] = swap(x[k+1], x[k+2])
    if y[k+2] > 1 or y[k+2] < 0:
        y[k+1], y[k+2] = swap(y[k+1], y[k+2])

In [34]:
X, Y = np.meshgrid(np.arange(N), np.arange(N))
plt.imshow(potential.cpu().numpy(), cmap='viridis', interpolation='nearest', origin='lower')
plt.scatter(100 * x, 100 * y, label='Particle Trajectory')